# Chroma and RAG

Store support articles as **vectors** (lists of numbers produced by an embedding model), save them in **Chroma**, then **retrieve** the articles closest to a user's question and let the LLM answer **only from those chunks**.

You do not need to implement similarity math yourself — Chroma's `similarity_search` does that when you index and query.


In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
dotenv_path = os.path.expanduser("~/dev.env")
load_dotenv(dotenv_path, override=True)
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in ~/dev.env"

## Embeddings

An **embedding model** turns text into a fixed-length vector. Phrases with similar meaning get vectors that are **close** in that space.

In this lab you only need one object — `OpenAIEmbeddings`. Chroma calls it when you `add_texts` (index) and when you `similarity_search` (retrieve).

In [2]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Behind the scenes: each KB article and each user question becomes a vector like this.
probe = embeddings.embed_query("database connection pool exhausted")
print("Model:", embeddings.model)
print("Vector length:", len(probe), "numbers per phrase")
print("Chroma compares question vectors to article vectors and returns the closest matches.")

Model: text-embedding-3-small
Vector length: 1536 numbers per phrase
Chroma compares question vectors to article vectors and returns the closest matches.


## Index the knowledge base

In [3]:
from langchain_chroma import Chroma

kb_dir = Path("data/kb")
texts, metas = [], []
for path in sorted(kb_dir.glob("*.md")):
    texts.append(path.read_text(encoding="utf-8"))
    metas.append({"kb_id": path.stem})
print("Loaded", len(texts), "files from", kb_dir)

store = Chroma(collection_name="kb_demo", embedding_function=embeddings)
store.add_texts(texts=texts, metadatas=metas)

Loaded 5 files from data\kb


['4b4094e1-0c52-4555-8f7e-753d36b90298',
 '041b2268-6eac-4e9e-816f-c0ca316e76ba',
 'ac8f9d66-7909-48f3-a03a-ba8b22844f2b',
 'a09ec42d-de47-418f-b4de-887251a6d55f',
 '45cf2a79-34d0-495b-8010-3fe7d3d63039']

## Retrieve

`similarity_search_with_relevance_scores` asks Chroma for the top `k` chunks. Higher scores mean a closer match to your question (Chroma computes this from the embedding vectors).

In [4]:
question = "Our app throws 500s and the logs say the pool timed out after 30 seconds"
hits = store.similarity_search_with_relevance_scores(question, k=2)
for doc, score in hits:
    print(f"[{doc.metadata['kb_id']}] score={round(score, 3)}")
    print(doc.page_content[:200], "...\n")

[kb-001-db-connection-pool] score=0.307
# KB-001: Database connection pool exhaustion

## Symptoms
- Application becomes slow or unresponsive under load.
- Users see "500 Internal Server Error" or "service unavailable".
- Requests time out  ...

[kb-005-upstream-timeout] score=0.1
# KB-005: Upstream/gateway timeouts (502/504)

## Symptoms
- Intermittent "502 Bad Gateway" or "504 Gateway Timeout".
- Some requests succeed; slow endpoints fail.

## Key error signatures
- `504 Gate ...



## Answer from context

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def rag_answer(question: str, k: int = 2) -> str:
    hits = store.similarity_search(question, k=k)
    context = "\n\n".join(f"[{h.metadata['kb_id']}] {h.page_content}" for h in hits)
    system = (
        "You are an L1 support assistant. Answer only from the CONTEXT. "
        "If the context does not contain the answer, say you do not know. "
        "List the KB ids you used."
    )
    user = f"CONTEXT:\n{context}\n\nQUESTION: {question}"
    return llm.invoke([("system", system), ("human", user)]).content

print(rag_answer(question))

The symptoms you are experiencing indicate that the application is facing database connection pool exhaustion. The specific error message you mentioned, "Connection is not available, request timed out after 30000ms," aligns with the key error signatures for this issue.

To resolve this, you can follow these steps:
1. Confirm current connections using the query: `SELECT count(*) FROM pg_stat_activity;`
2. Check for idle-in-transaction sessions and terminate any stuck ones.
3. Increase the pool size only after confirming that the database `max_connections` can support it.
4. Fix any connection leaks by ensuring every borrowed connection is returned in a `finally` block or context manager.
5. Consider adding a connection-pool metric and alert to catch this issue earlier in the future.

For prevention, set sensible pool min/max values and a connection `maxLifetime`, and perform load testing before major releases.

KB ids used: kb-001-db-connection-pool


In [6]:
print(rag_answer("How do I reset my office wifi router?"))

I do not know.
